# Practice Lab: Building MCP Servers & Clients (Lesson 10.5)

Module 10 · 8 exercises · FastMCP + Pydantic + composition + OpenAPI + testing + deployment

Exercises 1-5 run locally (pure Python simulations).


# Lesson 10.5: Building MCP Servers & Clients

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python simulations). Ex 6-8 are design.


In [ ]:
import json
import inspect


---
## Exercise 1 (Easy): FastMCP Hello World


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json, inspect

class FastMCPSim:
    def __init__(self, name): self.name=name; self.tools={}; self.resources={}; self.prompts={}
    def tool(self, func):
        hints=func.__annotations__; props={}; req=[]
        tm={str:"string",int:"integer",float:"number",bool:"boolean"}
        sig=inspect.signature(func)
        for p,th in hints.items():
            if p=="return": continue
            props[p]={"type":tm.get(th,"string"),"description":p}
            if sig.parameters[p].default is inspect.Parameter.empty: req.append(p)
        self.tools[func.__name__]={"name":func.__name__,"description":func.__doc__ or "",
            "inputSchema":{"type":"object","properties":props,"required":req,"additionalProperties":False},"func":func}
        return func
    def resource(self, uri):
        def dec(func): self.resources[uri]={"uri":uri,"name":func.__name__,"func":func}; return func
        return dec
    def prompt(self, func): self.prompts[func.__name__]={"name":func.__name__,"func":func}; return func

mcp = FastMCPSim("netsetos-support")
COURSES = {"genai":{"name":"GenAI Complete","price":14999},"python":{"name":"Python Mastery","price":9999},"cloud":{"name":"GCP Cloud","price":11999}}

@mcp.tool
def search_courses(topic: str) -> str:
    """Search Netsetos courses by topic."""
    c = COURSES.get(topic.lower())
    return json.dumps(c) if c else json.dumps({"error":f"No course: {topic}"})

@mcp.tool
def get_refund_policy(days_since: int) -> str:
    """Check refund eligibility."""
    if days_since<=7: return json.dumps({"eligible":True,"refund":"100%"})
    elif days_since<=30: return json.dumps({"eligible":True,"refund":"50%"})
    return json.dumps({"eligible":False,"refund":"0%"})

@mcp.tool
def calculate_emi(amount: int, months: int = 6) -> str:
    """Calculate monthly EMI."""
    return json.dumps({"emi":round(amount/months),"total":amount,"months":months})

@mcp.resource("netsetos://courses/catalog")
def catalog(): return json.dumps(COURSES)

@mcp.prompt
def support_greeting():
    """Support greeting template."""
    return "You are a Netsetos support agent..."

print("FastMCP Hello World:")
print(f"  Server: {mcp.name}")
print(f"  Tools: {len(mcp.tools)}")
for n,t in mcp.tools.items():
    s=t["inputSchema"]
    print(f"    @mcp.tool {n}({', '.join(s['properties'].keys())}), required={s['required']}")
print(f"  Resources: {len(mcp.resources)}")
for u in mcp.resources: print(f"    @mcp.resource('{u}')")
print(f"  Prompts: {len(mcp.prompts)}")

print(f"\n  Execution:")
print(f"    search_courses('genai') -> {search_courses('genai')}")
print(f"    get_refund_policy(5) -> {get_refund_policy(5)}")
print(f"    calculate_emi(14999,6) -> {calculate_emi(14999,6)}")
print(f"  FastMCP: 10 lines vs Raw SDK: 100+ lines")


</details>


---
## Exercise 2 (Easy): Pydantic Structured Output


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class WeatherResult:
    SCHEMA={"type":"object","properties":{"city":{"type":"string"},"temp_c":{"type":"number"},
        "condition":{"type":"string"},"humidity":{"type":"integer"}},"required":["city","temp_c","condition","humidity"]}
    def __init__(self,city,temp_c,cond,hum): self.city=city;self.temp_c=temp_c;self.cond=cond;self.hum=hum
    def to_dict(self): return {"city":self.city,"temp_c":self.temp_c,"condition":self.cond,"humidity":self.hum}
    def to_text(self): return f"{self.temp_c} C, {self.cond}, {self.hum}% in {self.city}"

class CourseResult:
    SCHEMA={"type":"object","properties":{"name":{"type":"string"},"price_inr":{"type":"integer"},
        "hours":{"type":"integer"},"rating":{"type":"number"}},"required":["name","price_inr","hours","rating"]}
    def __init__(self,name,price,hrs,rating): self.name=name;self.price=price;self.hrs=hrs;self.rating=rating
    def to_dict(self): return {"name":self.name,"price_inr":self.price,"hours":self.hrs,"rating":self.rating}
    def to_text(self): return f"{self.name}: {self.price} INR, {self.hrs}hrs, {self.rating}/5"

w = WeatherResult("Hyderabad", 34, "Sunny", 65)
c = CourseResult("GenAI Complete", 14999, 146, 4.8)

print("Pydantic Structured Output:")
print(f"\n  Weather:")
print(f"    outputSchema: {list(WeatherResult.SCHEMA['properties'].keys())}")
print(f"    structuredContent: {json.dumps(w.to_dict())}")
print(f"    content[]: {w.to_text()}")

print(f"\n  Course:")
print(f"    outputSchema: {list(CourseResult.SCHEMA['properties'].keys())}")
print(f"    structuredContent: {json.dumps(c.to_dict())}")
print(f"    content[]: {c.to_text()}")

for label, schema, data in [("Weather",WeatherResult.SCHEMA,w.to_dict()),("Course",CourseResult.SCHEMA,c.to_dict())]:
    ok = all(k in data for k in schema["required"])
    print(f"  [{label}] Schema valid: {ok}")

print(f"\n  Both returned: client chooses. Primitives wrapped: {{'result': value}}")


</details>


---
## Exercise 3 (Medium): Server Composition


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class MiniSrv:
    def __init__(self,name): self.name=name; self.tools={}
    def add(self,name,func,desc=""): self.tools[name]={"func":func,"desc":desc}

class Composed:
    def __init__(self,name): self.name=name; self.tools={}; self.mounts=[]
    def mount(self,srv,ns):
        self.mounts.append({"server":srv.name,"ns":ns})
        for tn,t in srv.tools.items():
            self.tools[f"{ns}_{tn}"]={"func":t["func"],"source":srv.name}
    def call(self,name,args):
        t=self.tools.get(name)
        return t["func"](**args) if t else {"error":f"Unknown: {name}"}

w = MiniSrv("weather"); w.add("get_forecast",lambda city,days=3:{"city":city,"days":days,"outlook":"Sunny"})
w.add("get_alerts",lambda region:{"region":region,"alerts":0})
c = MiniSrv("courses"); c.add("search",lambda topic:{"name":f"{topic.title()} Course","price":14999})
c.add("get_syllabus",lambda course_id:{"modules":14,"hours":146})

main = Composed("netsetos-main")
main.mount(w,"weather"); main.mount(c,"courses")

print("Server Composition:")
print(f"  Main: {main.name}, {len(main.mounts)} sub-servers")
for m in main.mounts: print(f"    {m['ns']}/ -> {m['server']}")
print(f"\n  Tools ({len(main.tools)}):")
for n in main.tools: print(f"    {n}")

r1=main.call("weather_get_forecast",{"city":"Hyderabad","days":5})
r2=main.call("courses_search",{"topic":"genai"})
print(f"\n  weather_get_forecast -> {r1}")
print(f"  courses_search -> {r2}")
print(f"\n  Advantages: no collisions, independent updates, reusable")


</details>


---
## Exercise 4 (Medium): OpenAPI to MCP Conversion


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def openapi_to_mcp(spec):
    tools=[]
    for path,methods in spec.get("paths",{}).items():
        for method,det in methods.items():
            props={}; req=[]
            for p in det.get("parameters",[]):
                loc=p.get("in","query"); nm=p["name"]
                key=f"path_{nm}" if loc=="path" else nm
                props[key]={"type":p.get("schema",{}).get("type","string"),"description":p.get("description",f"{nm}")}
                if p.get("required"): req.append(key)
            body=det.get("requestBody",{})
            if body:
                bp=(body.get("content",{}).get("application/json",{}).get("schema",{}).get("properties",{}))
                br=(body.get("content",{}).get("application/json",{}).get("schema",{}).get("required",[]))
                for k,v in bp.items(): props[f"body_{k}"]=v
                for k in br: req.append(f"body_{k}")
            schema={"type":"object","properties":props,"required":req,"additionalProperties":False} if props else {"type":"object","additionalProperties":False}
            tools.append({"name":det.get("operationId",f"{method}_{path}"),"description":det.get("summary",""),"inputSchema":schema})
    return tools

spec={"paths":{
    "/courses/{id}":{"get":{"operationId":"get_course","summary":"Get course by ID",
        "parameters":[{"name":"id","in":"path","required":True,"schema":{"type":"string"}},
                      {"name":"reviews","in":"query","schema":{"type":"boolean"}}]}},
    "/courses/search":{"get":{"operationId":"search_courses","summary":"Search courses",
        "parameters":[{"name":"q","in":"query","required":True,"schema":{"type":"string"},"description":"Search query"}]}},
    "/enrollments":{"post":{"operationId":"create_enrollment","summary":"Enroll student",
        "requestBody":{"content":{"application/json":{"schema":{"type":"object",
            "properties":{"course_id":{"type":"string"},"email":{"type":"string"}},"required":["course_id","email"]}}}}}}}}

tools=openapi_to_mcp(spec)
print("OpenAPI -> MCP Conversion:")
for t in tools:
    props=list(t["inputSchema"].get("properties",{}).keys())
    req=t["inputSchema"].get("required",[])
    print(f"  {t['name']}: params={props}, required={req}")
print(f"\nRouteMap: GET->tool, GET /stats->resource, DELETE->exclude")
print(f"Portable: flat schemas work across all providers")


</details>


---
## Exercise 5 (Medium): In-Memory Test Suite


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class MCPTest:
    def __init__(self,tools): self.tools=tools; self.results=[]
    def call(self,name,args):
        if name not in self.tools: return {"error":f"Unknown: {name}","isError":True}
        try: return {"content":[{"type":"text","text":self.tools[name](**args)}],"isError":False}
        except Exception as e: return {"content":[{"type":"text","text":str(e)}],"isError":True}
    def test(self,name,label,args,expect_in=None,expect_err=False):
        r=self.call(name,args); err=r.get("isError",False)
        ok=True; reason="OK"
        if expect_err and not err: ok=False; reason="Expected error"
        elif not expect_err and err: ok=False; reason=f"Unexpected error"
        elif expect_in:
            txt=r.get("content",[{}])[0].get("text","")
            if expect_in not in txt: ok=False; reason=f"'{expect_in}' not found"
        self.results.append({"test":label,"ok":ok})
        print(f"    [{'PASS' if ok else 'FAIL'}] {label}: {reason}")

tools={"search":lambda topic: json.dumps({"name":"GenAI","price":14999}) if topic.lower() in ["genai","python"] else json.dumps({"error":f"No: {topic}"}),
       "refund":lambda days: json.dumps({"eligible":days<=7,"refund":"100%" if days<=7 else "50%" if days<=30 else "0%"}),
       "emi":lambda amount,months: json.dumps({"emi":round(amount/months),"total":amount})}

t=MCPTest(tools)
print("In-Memory MCP Test Suite:")
print("  1. Tool Execution:")
t.test("search","search_valid",{"topic":"genai"},"GenAI")
t.test("search","search_invalid",{"topic":"blockchain"},"No:")
t.test("refund","refund_ok",{"days":5},"100%")
t.test("refund","refund_expired",{"days":45},"0%")
t.test("emi","emi_calc",{"amount":14999,"months":6},"2500")
print("  2. Error Handling:")
t.test("unknown","unknown_tool",{},expect_err=True)

passed=sum(1 for r in t.results if r["ok"])
print(f"\n  Results: {passed}/{len(t.results)} ({passed/len(t.results):.0%})")
print(f"\n  RULE: stdio stdout = JSON-RPC ONLY. Use ctx.info() not print()")
print(f"  CI: pytest tests/ --asyncio-mode=auto")


</details>


---
## Exercise 6 (Challenge): Cloud Run Deployment
Design/architecture. See practice-lab-10_5.html.


In [ ]:
# Exercise 6: Cloud Run Deployment
pass


<details><summary>Solution</summary>


In [ ]:
print('Cloud Run: FastMCP + streamable-http + stateless_http=True')
print('Deploy: gcloud run deploy --source . --region asia-south1')
print('Health: @mcp.custom_route(/health). OTel: native spans. Free: 2M req/mo')


</details>


---
## Exercise 7 (Challenge): Bhashini MCP Server
Design/architecture. See practice-lab-10_5.html.


In [ ]:
# Exercise 7: Bhashini MCP Server
pass


<details><summary>Solution</summary>


In [ ]:
print('Bhashini MCP: 4 tools (translate, STT, TTS, OCR) x 22 languages')
print('3-step API (search->config->compute) abstracted to 1 tool call')
print('Impact: 58 clients x 22 langs x 4 tools = 5,104 capabilities')


</details>


---
## Exercise 8 (Challenge): Multi-Host Configuration
Design/architecture. See practice-lab-10_5.html.


In [ ]:
# Exercise 8: Multi-Host Configuration
pass


<details><summary>Solution</summary>


In [ ]:
print('Claude/Cursor: mcpServers key. VS Code: servers key (DIFFERENT!)')
print('Cursor: ~40 tool limit (silently drops!). Windsurf: 100')
print('VS Code: inputs array for secure credential prompting (best)')


</details>
